Import Testset

In [91]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import Subset
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F
import os
import onnx
from onnx2torch import convert
import numpy as np
import onnxruntime as ort
from onnx import checker, helper
from onnx_pytorch import code_gen
from onnxsim import simplify

In [92]:


def get_loss_accuracy_onnx(onnx_model_path, criterion, test_loader, num_classes):
    # Crea una sessione di inferenza per il modello ONNX
    
    #print(onnx_model_path)
    ort_session = ort.InferenceSession(onnx_model_path)
    input_name = ort_session.get_inputs()[0].name

    running_test_loss = 0.0
    correct_test = 0
    total_test = 0

    with torch.no_grad():
        for inputs, targets in test_loader:
            # Porta i dati a numpy per l'inferenza ONNX
            inputs = inputs.view(inputs.size(0), -1).numpy().astype(np.float32)
            targets = targets.numpy()

            # Inferenza con ONNX
            outputs = ort_session.run(None, {input_name: inputs})
            outputs = torch.tensor(outputs[0])  # Converte l'output in tensore PyTorch per compatibilità

            # Calcola la perdita
            if isinstance(criterion, torch.nn.MSELoss):
                targets_hot_encoded = F.one_hot(torch.tensor(targets), num_classes=num_classes).float()
                loss = criterion(outputs, targets_hot_encoded)
                running_test_loss += loss.item()
            else:
                targets_tensor = torch.tensor(targets).long()  # Necessario per criteri come CrossEntropyLoss
                loss = criterion(outputs, targets_tensor)
                running_test_loss += loss.item()

            # Calcola l'accuratezza
            _, predicted = torch.max(outputs.data, 1)
            total_test += targets.size
            correct_test += (predicted.numpy() == targets).sum()

    # Calcola perdita media e accuratezza per l'epoca corrente
    test_loss = running_test_loss / len(test_loader)
    test_accuracy = 100 * correct_test / total_test
    return test_loss, test_accuracy


In [93]:
path = r'D:\11-10-24\deep_test\results\neuron_pruning'
folder_path = r'C:\Users\andr3\Desktop\pyNeVer-20240815T131503Z-001\pyNeVer\examples\converted'
test_dim = 4000
train_dim = 10000
test_batch_size = 64
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: torch.flatten(x))])
test_set = datasets.MNIST(root='././data', train=False, download=True, transform=transform)

# Create a subset
test_subset = Subset(test_set, range(test_dim))  # Take the first test_dim elements from the test set

test_loader = DataLoader(test_subset, batch_size=test_batch_size, shuffle=False)

In [94]:
pandas_like_list = []
for file in os.listdir(path):
    # Controlla se l'elemento è un file con estensione .onnx
    if file.endswith('.onnx'):
        onnx_model_path = os.path.join(path, file)  
        
        #Calcolo la test e train loss
        test_loss, test_accuracy = get_loss_accuracy_onnx(onnx_model_path, criterion, test_loader, 10)
        temp_dict = {"file": file, "test_loss": test_loss, "test_accuracy": test_accuracy}
        
        pandas_like_list.append(temp_dict)
        print(f"file: {file}, test_loss: {test_loss}, test_accuracy: {test_accuracy}")
        
        
        
            

Fail: [ONNXRuntimeError] : 1 : FAIL : Load model from D:\11-10-24\deep_test\results\neuron_pruning\ns_pruned_200_1.onnx failed:Type Error: Type parameter (T) of Optype (Gemm) bound to different types (tensor(double) and tensor(float) in node ().